In [1]:
import os
import json
import datetime
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
    
engine = create_engine('mysql+pymysql://root@localhost:3306/music_development')
music = engine.connect()
data_path = '../data/'

In [2]:
def scan_mp4_files(folder_path):
    """Scan folder for MP4 files and return sorted list"""
    mp4_files = []
    
    try:
        for file in os.listdir(folder_path):
            if file.lower().endswith('.mp4'):
                mp4_files.append(file)
        
        # Sort alphabetically
        mp4_files.sort()
        
    except Exception as e:
        print(f"Error scanning folder: {e}")
    
    return mp4_files

In [3]:
def generate_songs_json(mp4_files, output_file='songs.json'):
    """Generate JSON file with MP4 file list"""
    
    data = {
        "generated_at": datetime.datetime.now().isoformat(),
        "total_songs": len(mp4_files),
        "songs": mp4_files
    }
    
    try:
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        print(f"✅ Successfully generated {output_file}")
        print(f"📊 Total songs: {len(mp4_files)}")
        print(f"🕒 Generated at: {data['generated_at']}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error generating JSON file: {e}")
        return False

In [4]:
def analyze_filenames(mp4_files):
    """Analyze MP4 filenames and extract song/artist information"""
    
    print("\n🎵 Filename Analysis:")
    
    for filename in mp4_files:
        # Remove .mp4 extension
        name_without_ext = filename[:-4]
        
        # Try to split by underscore
        if '_' in name_without_ext:
            parts = name_without_ext.split('_')
            song_title = parts[0].strip()
            # artist = ' '.join(parts[1:]).strip()
            # print(f"   🎶 '{song_title}' by {artist}")
            print(f"'{song_title}'")
        else:
            print(f"   📁 {name_without_ext} (no artist)")

In [5]:
def generate_upload_checklist(mp4_files, lyrics_folder):
    """Generate a checklist of files to upload to AWS"""
    
    print("\n📋 Upload Checklist for AWS:")
    print("=" * 50)
    
    # Core app files
    core_files = ['index.html', 'style.css', 'script.js', 'songs.json']
    
    print("\n🚀 Core App Files:")
    for file in core_files:
        if os.path.exists(file):
            print(f"   ✅ {file}")
        else:
            print(f"   ❌ {file} (MISSING)")
    
    print(f"\n🎵 MP4 Files to Upload ({len(mp4_files)} files):")
    for i, mp4_file in enumerate(mp4_files, 1):
        file_path = os.path.join(lyrics_folder, mp4_file)
        if os.path.exists(file_path):
            file_size = os.path.getsize(file_path) / (1024 * 1024)  # Size in MB
            print(f"   ✅ {i:2d}. {mp4_file} ({file_size:.1f} MB)")
        else:
            print(f"   ❌ {i:2d}. {mp4_file} (FILE NOT FOUND)")
    
    print("\n💡 Upload all checked files to your AWS Lyrics folder")

In [6]:
def main():
    """Main function to run the MP4 file scanner"""
    
    # Set your Lyrics folder path - CHANGE THIS TO YOUR MP4 FOLDER
    LYRICS_FOLDER = r"C:\Users\PC1\OneDrive\A5\Data\Videos\AWS"   #chgfld  
    
    print(f"🎵 MP4 File Scanner")
    print(f"📁 Scanning folder: {LYRICS_FOLDER}")
    print("=" * 50)
    
    # Scan for MP4 files
    mp4_files = scan_mp4_files(LYRICS_FOLDER)
    
    if not mp4_files:
        print("❌ No MP4 files found in the specified folder.")
        return
    
    print(f"\n✅ Found {len(mp4_files)} MP4 files:")
    for i, file in enumerate(mp4_files, 1):
        print(f"   {i:2d}. {file}")
    
    # Generate songs.json
    generate_songs_json(mp4_files)
    
    # Analyze filenames
    analyze_filenames(mp4_files)
    
    # Generate upload checklist
    generate_upload_checklist(mp4_files, LYRICS_FOLDER)
    
    print("\n" + "=" * 50)
    print("🎉 Script completed successfully!")


In [7]:
def analyze_filenames(mp4_files):
    """Extract song titles from MP4 filenames into DataFrame"""
    
    song_titles = []
    
    for filename in mp4_files:
        # Remove .mp4 extension
        name_without_ext = filename[:-4]
        
        # Extract song title (before first underscore if exists)
        if '_' in name_without_ext:
            song_title = name_without_ext.split('_')[0].strip()
        else:
            song_title = name_without_ext
        
        song_titles.append(song_title)
    
    # Create DataFrame with song_title column
    return pd.DataFrame({'song_title': song_titles})

In [8]:
# Call the function
df = analyze_filenames(mp4_files)

# Display the DataFrame
print("\n📊 DataFrame:")
print(df)

# Access just the song_title column
print("\n🎵 Song Titles:")
print(df['song_title'])

NameError: name 'mp4_files' is not defined

In [9]:
# Direct scanning approach
LYRICS_FOLDER = r"C:\Users\PC1\OneDrive\A5\Data\Videos\AWS"

# Scan for MP4 files
mp4_files = []
for file in os.listdir(LYRICS_FOLDER):
    if file.lower().endswith('.mp4'):
        mp4_files.append(file)
mp4_files.sort()

# Create DataFrame
df = analyze_filenames(mp4_files)
print("\n📊 DataFrame:")
print(df)


📊 DataFrame:
                            song_title
0    (Everything I Do) I Do It For You
1               A Change Is Gonna Come
2                    A Day in the Life
3                     Against the Wind
4                         All Too Well
..                                 ...
104                   You're Beautiful
105                You've got a friend
106                           Your Man
107               You’re Still The One
108                             Zombie

[109 rows x 1 columns]


In [11]:
sql = '''
SELECT * FROM songs
'''
df_songs = pd.read_sql(sql, music)
df_songs.sample(5)

,id,rank,name,artist_id,length,location,genre_id,youtube_code,released_date,rating,website
49,52,42,Waterloo Sunset,46,193,Waterloo_Sunset.mp3,26,N_MqfF0WBsU,1968-02-25,2,0
96,99,99,Fortunate Son,25,141,Fortunate_Son.mp3,12,ZWijx_AgPiA,1969-10-09,3,0
333,344,146,Everyday People,83,300,Everyday_People.mp3,26,YUUhDoCx8zc,1968-11-03,1,0
27,30,28,A Day in the Life,1,336,The_Beatles_-_A_Day_In_The_Life.mp3,2,usNsCeOV4GM,1967-06-01,5,0
324,335,483,White Rabbit,117,300,White_Rabbit.mp3,26,EUY2kJE0AZE,1967-09-02,2,0


In [13]:
df_merge = pd.merge(df,df_songs,on=[song_title,name])
df_merge

NameError: name 'song_title' is not defined